# UMUD Challenge — Muscle Architecture from Ultrasound
### Strategy: diverse-but-light dual ensemble (ResNet34-UNet + ResNet50-DeepLabV3) at 384px, geometry-safe augmentation, checkpointed + resumable, incremental submission writes.

**Why this design (research + your own submission history):**
- Published muscle-architecture pipelines (Cronin et al. 2020; DL_Track_US, Ritsche et al.) all use **U-Net-family encoders at modest resolution**, not huge single models — this matches your Submission 3 (best) and explains why Submission 4 (bigger EfficientNet-B3 @512px, heavy elastic/grid augmentation) regressed.
- Augmentation helps *when it doesn't destroy tissue geometry* — so we use only flip / mild affine / photometric noise, **never elastic or grid distortion**.
- We get robustness from a **small ensemble of two architecturally different, still-lightweight models** (ResNet34-UNet, ResNet50-DeepLabV3) + horizontal-flip TTA, instead of one bigger model.
- Every stage below checkpoints to disk, is resumable, and writes a valid `submission.csv` as soon as Stage A finishes.


## STEP 0 — Environment setup, warning suppression, reproducibility

In [ ]:

import os, sys, time, glob, math, json, random, pickle, warnings
os.environ["OPENCV_LOG_LEVEL"] = "SILENT"
warnings.filterwarnings("ignore")

import cv2
cv2.setLogLevel(0)

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import albumentations as A
from albumentations.pytorch import ToTensorV2

import torchvision
from torchvision.models import ResNet34_Weights
from torchvision.models.segmentation import deeplabv3_resnet50, DeepLabV3_ResNet50_Weights

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
n_gpu = torch.cuda.device_count()
print(f"[STEP 0] device={device}  n_gpu={n_gpu}  torch={torch.__version__}  cv2={cv2.__version__}  albumentations={A.__version__}")

WORK_DIR = "/kaggle/working"
CKPT_DIR = os.path.join(WORK_DIR, "checkpoints")
CACHE_DIR = os.path.join(WORK_DIR, "pred_cache")
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)
print(f"[STEP 0] checkpoints -> {CKPT_DIR}")
print(f"[STEP 0] prediction cache -> {CACHE_DIR}")

T0 = time.time()
TIME_BUDGET_HOURS = 4.0
TRAIN_TIME_BUDGET_END = T0 + TIME_BUDGET_HOURS * 3600.0

def elapsed_min():
    return (time.time() - T0) / 60.0

print(f"[STEP 0] training time budget = {TIME_BUDGET_HOURS} h (wall clock from now)")


## STEP 1 — Locate the dataset
**If any path prints as `None`, edit `MANUAL_OVERRIDE` below with the correct absolute path.**

In [ ]:

INPUT_ROOT = "/kaggle/input"

def find_dir(root, target_name):
    for dirpath, dirnames, _ in os.walk(root):
        if os.path.basename(dirpath).lower() == target_name.lower():
            return dirpath
    return None

def find_file(root, target_name):
    for dirpath, _, filenames in os.walk(root):
        for f in filenames:
            if f.lower() == target_name.lower():
                return os.path.join(dirpath, f)
    return None

AUTO = {
    "apo_imgs_v1":  find_dir(INPUT_ROOT, "apo_imgs_v1"),
    "apo_masks_v1": find_dir(INPUT_ROOT, "apo_masks_v1"),
    "fasc_imgs_v1": find_dir(INPUT_ROOT, "fasc_imgs_v1"),
    "fasc_masks_v1":find_dir(INPUT_ROOT, "fasc_masks_v1"),
    "test_dir":  find_dir(INPUT_ROOT, "test_set_v2"),
    "sample_csv": find_file(INPUT_ROOT, "sample_submission.csv"),
}

MANUAL_OVERRIDE = {
    # "apo_imgs_v1":  "/kaggle/input/.../apo_imgs",
    # "apo_masks_v1": "/kaggle/input/.../apo_masks",
    # "fasc_imgs_v1": "/kaggle/input/.../fasc_imgs",
    # "fasc_masks_v1":"/kaggle/input/.../fasc_masks",
    # "test_dir":  "/kaggle/input/.../test_images_v2/test_set_v2",
    # "sample_csv":"/kaggle/input/.../sample_submission.csv",
}
AUTO.update({k: v for k, v in MANUAL_OVERRIDE.items() if v})

for k, v in AUTO.items():
    n = len(glob.glob(os.path.join(v, "*"))) if (v and os.path.isdir(v)) else ("-" if v else 0)
    print(f"[STEP 1] {k:12s} -> {v}   (n_items={n})")

APO_IMG_DIR, APO_MASK_DIR = AUTO["apo_imgs_v1"], AUTO["apo_masks_v1"]
FASC_IMG_DIR, FASC_MASK_DIR = AUTO["fasc_imgs_v1"], AUTO["fasc_masks_v1"]
TEST_DIR = AUTO["test_dir"]
SAMPLE_CSV = AUTO["sample_csv"]

assert APO_IMG_DIR and APO_MASK_DIR and FASC_IMG_DIR and FASC_MASK_DIR and TEST_DIR and SAMPLE_CSV, \
    "[STEP 1] One or more paths are missing — fill MANUAL_OVERRIDE above and re-run this cell."


## STEP 2 — Build matched (image, mask) file lists

In [ ]:

def match_pairs(img_dir, mask_dir):
    img_files = sorted(glob.glob(os.path.join(img_dir, "*")))
    pairs = []
    for ip in img_files:
        stem = os.path.splitext(os.path.basename(ip))[0]
        cands = glob.glob(os.path.join(mask_dir, stem + ".*"))
        if cands:
            pairs.append((ip, cands[0]))
    return pairs

apo_pairs = match_pairs(APO_IMG_DIR, APO_MASK_DIR)
fasc_pairs = match_pairs(FASC_IMG_DIR, FASC_MASK_DIR)
print(f"[STEP 2] apo pairs matched:  {len(apo_pairs)} / {len(glob.glob(os.path.join(APO_IMG_DIR,'*')))} images")
print(f"[STEP 2] fasc pairs matched: {len(fasc_pairs)} / {len(glob.glob(os.path.join(FASC_IMG_DIR,'*')))} images")
assert len(apo_pairs) > 10 and len(fasc_pairs) > 10, "[STEP 2] Too few matched pairs — check filename conventions."

random.Random(SEED).shuffle(apo_pairs)
random.Random(SEED).shuffle(fasc_pairs)

def split(pairs, val_frac=0.15):
    n_val = max(1, int(len(pairs) * val_frac))
    return pairs[n_val:], pairs[:n_val]

apo_train, apo_val = split(apo_pairs)
fasc_train, fasc_val = split(fasc_pairs)
print(f"[STEP 2] apo  -> train={len(apo_train)} val={len(apo_val)}")
print(f"[STEP 2] fasc -> train={len(fasc_train)} val={len(fasc_val)}")


## STEP 3 — Dataset / augmentation
Geometry-preserving only: flip / mild affine / photometric noise. **No elastic/grid distortion.**
`_albu_gauss_noise` below auto-adapts to whichever Albumentations API (`var_limit` vs `std_range`) is installed on the Kaggle image, so the notebook doesn't crash on a version mismatch.

In [ ]:

IMG_SIZE = 384
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

def _albu_gauss_noise(var_limit=(5.0, 20.0), p=0.3):
    try:
        return A.GaussNoise(var_limit=var_limit, p=p)
    except TypeError:
        lo = (var_limit[0] ** 0.5) / 255.0
        hi = (var_limit[1] ** 0.5) / 255.0
        return A.GaussNoise(std_range=(lo, hi), p=p)

def get_train_aug():
    return A.Compose([
        A.Resize(IMG_SIZE, IMG_SIZE),
        A.HorizontalFlip(p=0.5),
        A.Affine(scale=(0.95, 1.05), translate_percent=(0.0, 0.03), rotate=(-6, 6),
                  shear=(-3, 3), p=0.5, mode=cv2.BORDER_CONSTANT, cval=0,
                  mask_mode=cv2.BORDER_CONSTANT, cval_mask=0),
        A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.5),
        _albu_gauss_noise(var_limit=(5.0, 20.0), p=0.3),
        A.Sharpen(alpha=(0.1, 0.25), lightness=(0.9, 1.1), p=0.2),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2(),
    ])

def get_val_aug():
    return A.Compose([
        A.Resize(IMG_SIZE, IMG_SIZE),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2(),
    ])

class SegDataset(Dataset):
    def __init__(self, pairs, aug):
        self.pairs = pairs
        self.aug = aug

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        img_path, mask_path = self.pairs[idx]
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        if img is None or mask is None:
            raise RuntimeError(f"Failed to read {img_path} or {mask_path}")
        if mask.shape[:2] != img.shape[:2]:
            mask = cv2.resize(mask, (img.shape[1], img.shape[0]), interpolation=cv2.INTER_NEAREST)
        img_rgb = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
        mask_bin = (mask > 127).astype(np.uint8)
        out = self.aug(image=img_rgb, mask=mask_bin)
        m = out["mask"].float()
        if m.dim() == 2:
            m = m.unsqueeze(0)
        return out["image"], m

BATCH_SIZE = 16
NUM_WORKERS = 2

def make_loaders(train_pairs, val_pairs):
    train_ds = SegDataset(train_pairs, get_train_aug())
    val_ds = SegDataset(val_pairs, get_val_aug())
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                               num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                             num_workers=NUM_WORKERS, pin_memory=True)
    return train_loader, val_loader

apo_train_loader, apo_val_loader = make_loaders(apo_train, apo_val)
fasc_train_loader, fasc_val_loader = make_loaders(fasc_train, fasc_val)
print("[STEP 3] loaders ready")


## STEP 4 — Two lightweight, architecturally-different models

In [ ]:

class ResNet34UNet(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        weights = ResNet34_Weights.IMAGENET1K_V1 if pretrained else None
        resnet = torchvision.models.resnet34(weights=weights)
        self.enc0 = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu)
        self.pool0 = resnet.maxpool
        self.enc1 = resnet.layer1
        self.enc2 = resnet.layer2
        self.enc3 = resnet.layer3
        self.enc4 = resnet.layer4

        def block(cin, cout):
            return nn.Sequential(
                nn.Conv2d(cin, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
                nn.Conv2d(cout, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
            )
        self.up4 = nn.ConvTranspose2d(512, 256, 2, stride=2); self.dec4 = block(512, 256)
        self.up3 = nn.ConvTranspose2d(256, 128, 2, stride=2); self.dec3 = block(256, 128)
        self.up2 = nn.ConvTranspose2d(128, 64, 2, stride=2);  self.dec2 = block(128, 64)
        self.up1 = nn.ConvTranspose2d(64, 64, 2, stride=2);   self.dec1 = block(128, 64)
        self.up0 = nn.ConvTranspose2d(64, 32, 2, stride=2);   self.dec0 = block(32, 32)
        self.head = nn.Conv2d(32, 1, kernel_size=1)

    def forward(self, x):
        e0 = self.enc0(x)
        p0 = self.pool0(e0)
        e1 = self.enc1(p0)
        e2 = self.enc2(e1)
        e3 = self.enc3(e2)
        e4 = self.enc4(e3)
        d4 = self.dec4(torch.cat([self.up4(e4), e3], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), e2], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e1], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e0], dim=1))
        d0 = self.dec0(self.up0(d1))
        return self.head(d0)


def build_deeplab(pretrained=True):
    weights = DeepLabV3_ResNet50_Weights.COCO_WITH_VOC_LABELS_V1 if pretrained else None
    model = deeplabv3_resnet50(weights=weights, aux_loss=False)
    model.classifier[4] = nn.Conv2d(256, 1, kernel_size=1)
    return model


class DeepLabWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, x):
        return self.model(x)["out"]

print("[STEP 4] model builders defined")


## STEP 5 — Loss, checkpointed / resumable training loop

In [ ]:

class BCEDiceLoss(nn.Module):
    def __init__(self, bce_weight=0.5, smooth=1.0):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
        self.bce_weight = bce_weight
        self.smooth = smooth

    def forward(self, logits, targets):
        bce = self.bce(logits, targets)
        probs = torch.sigmoid(logits)
        inter = (probs * targets).sum(dim=(1, 2, 3))
        union = probs.sum(dim=(1, 2, 3)) + targets.sum(dim=(1, 2, 3))
        dice_loss = 1 - ((2 * inter + self.smooth) / (union + self.smooth)).mean()
        return self.bce_weight * bce + (1 - self.bce_weight) * dice_loss


def train_model(model_name, model, train_loader, val_loader, epochs=30, lr=1e-3, patience=6):
    ckpt_path = os.path.join(CKPT_DIR, f"{model_name}.pt")
    best_path = os.path.join(CKPT_DIR, f"{model_name}_best.pt")

    model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))
    criterion = BCEDiceLoss()

    start_epoch, best_dice, bad_epochs = 0, -1.0, 0
    if os.path.exists(ckpt_path):
        ck = torch.load(ckpt_path, map_location=device)
        model.load_state_dict(ck["model"])
        opt.load_state_dict(ck["opt"])
        scheduler.load_state_dict(ck["sched"])
        start_epoch = ck["epoch"] + 1
        best_dice = ck.get("best_dice", -1.0)
        bad_epochs = ck.get("bad_epochs", 0)
        print(f"[{model_name}] RESUMING from epoch {start_epoch} (best_dice so far = {best_dice:.4f})")
    else:
        print(f"[{model_name}] starting fresh training run ({epochs} epochs max)")

    for epoch in range(start_epoch, epochs):
        if time.time() > TRAIN_TIME_BUDGET_END:
            print(f"[{model_name}] TIME BUDGET REACHED before epoch {epoch+1} -> stopping, using last checkpoint.")
            break

        model.train()
        running_loss = 0.0
        for imgs, masks in train_loader:
            imgs, masks = imgs.to(device, non_blocking=True), masks.to(device, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
                logits = model(imgs)
                loss = criterion(logits, masks)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            running_loss += loss.item() * imgs.size(0)
        train_loss = running_loss / len(train_loader.dataset)

        model.eval()
        val_dice_sum = 0.0
        with torch.no_grad():
            for imgs, masks in val_loader:
                imgs, masks = imgs.to(device), masks.to(device)
                with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
                    logits = model(imgs)
                probs = torch.sigmoid(logits)
                preds = (probs > 0.5).float()
                inter = (preds * masks).sum(dim=(1, 2, 3))
                union = preds.sum(dim=(1, 2, 3)) + masks.sum(dim=(1, 2, 3))
                dice = ((2 * inter + 1) / (union + 1)).mean().item()
                val_dice_sum += dice * imgs.size(0)
        val_dice = val_dice_sum / len(val_loader.dataset)
        scheduler.step()

        print(f"[{model_name}] epoch {epoch+1}/{epochs}  train_loss={train_loss:.4f}  "
              f"val_dice={val_dice:.4f}  elapsed={elapsed_min():.1f}min")

        torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "sched": scheduler.state_dict(),
                    "epoch": epoch, "best_dice": best_dice, "bad_epochs": bad_epochs}, ckpt_path)

        if val_dice > best_dice:
            best_dice = val_dice
            bad_epochs = 0
            torch.save(model.state_dict(), best_path)
            print(f"[{model_name}]   -> new best (dice={best_dice:.4f}), saved to {best_path}")
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                print(f"[{model_name}] early stopping (no val_dice improvement for {patience} epochs)")
                break

    if os.path.exists(best_path):
        model.load_state_dict(torch.load(best_path, map_location=device))
        print(f"[{model_name}] loaded best checkpoint (dice={best_dice:.4f}) for inference")
    return model, best_dice

print("[STEP 5] training loop defined")


## STEP 6 — Geometry-extraction function (unchanged, as supplied)

In [ ]:

SCALE_MM_PER_PIXEL = 0.06979

def measure_geometry(apo_mask, fasc_mask, scale_mm_per_pixel=SCALE_MM_PER_PIXEL):
    n_labels, labels, stats, _ = cv2.connectedComponentsWithStats(fasc_mask, 8)
    angles, weights = [], []
    for i in range(1, n_labels):
        area = stats[i, cv2.CC_STAT_AREA]
        if area < 50:
            continue
        pts = np.column_stack(np.where(labels == i)).astype(np.float64)
        if len(pts) < 5:
            continue
        cov = np.cov(pts.T)
        if cov.shape != (2, 2):
            continue
        evals, evecs = np.linalg.eigh(cov)
        principal = evecs[:, np.argmax(evals)]
        angle = np.degrees(np.arctan2(abs(principal[0]), abs(principal[1])))
        angles.append(angle)
        weights.append(float(area))
    pa_deg = np.average(angles, weights=weights) if angles else 15.0

    n_apo, apo_labels, apo_stats, _ = cv2.connectedComponentsWithStats(apo_mask, 8)
    if n_apo >= 3:
        comps = sorted([(i, apo_stats[i, cv2.CC_STAT_AREA]) for i in range(1, n_apo)], key=lambda x: x[1], reverse=True)
        c1, c2 = comps[0][0], comps[1][0]
        y1, y2 = np.mean(np.where(apo_labels == c1)[0]), np.mean(np.where(apo_labels == c2)[0])
        sup_c, deep_c = (c1, c2) if y1 < y2 else (c2, c1)
        sup_mask = (apo_labels == sup_c).astype(np.uint8)
        deep_mask = (apo_labels == deep_c).astype(np.uint8)
        sup_y = np.argmax(sup_mask[::-1], axis=0)
        sup_bottom = np.where(sup_mask.any(axis=0), sup_mask.shape[0] - 1 - sup_y, np.nan)
        deep_top = np.where(deep_mask.any(axis=0), np.argmax(deep_mask, axis=0), np.nan)
        thickness_profile = deep_top - sup_bottom
        valid = ~np.isnan(thickness_profile) & (thickness_profile > 0)
        mt_mm = np.nanmedian(thickness_profile[valid]) * scale_mm_per_pixel if np.sum(valid) > 50 else 20.0
    else:
        mt_mm = 20.0

    fl_mm = mt_mm / math.sin(math.radians(max(pa_deg, 3.0)))

    pa_deg = float(np.clip(pa_deg, 3.0, 45.0))
    fl_mm = float(np.clip(fl_mm, 20.0, 200.0))
    mt_mm = float(np.clip(mt_mm, 5.0, 40.0))
    return pa_deg, fl_mm, mt_mm

print("[STEP 6] geometry extraction ready")


## STEP 7 — Ensemble + TTA inference helper

In [ ]:

_val_aug = get_val_aug()

def predict_mask_ensemble(models, img_path, tta_flip=True):
    img0 = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    if img0 is None:
        raise RuntimeError(f"Could not read {img_path}")
    h0, w0 = img0.shape[:2]
    img_rgb = cv2.cvtColor(img0, cv2.COLOR_GRAY2RGB)
    x = _val_aug(image=img_rgb)["image"].unsqueeze(0).to(device)

    variants = [x]
    if tta_flip:
        variants.append(torch.flip(x, dims=[3]))

    probs_sum, n = None, 0
    with torch.no_grad(), torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
        for model in models:
            model.eval()
            for i, v in enumerate(variants):
                logits = model(v)
                p = torch.sigmoid(logits)
                if i == 1:
                    p = torch.flip(p, dims=[3])
                probs_sum = p if probs_sum is None else probs_sum + p
                n += 1
    probs = (probs_sum / n).squeeze().float().cpu().numpy()
    probs_full = cv2.resize(probs, (w0, h0), interpolation=cv2.INTER_LINEAR)
    mask = (probs_full > 0.5).astype(np.uint8)
    kernel = np.ones((3, 3), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    return mask

print("[STEP 7] inference helper ready")


## STEP 8 — Submission builder (resumable, incremental writes)

In [ ]:

raw_sample = pd.read_csv(SAMPLE_CSV, sep=None, engine="python")
print(f"[STEP 8] sample_submission columns: {raw_sample.columns.tolist()}")
print(raw_sample.head(3))

ID_COL = raw_sample.columns[0]
TARGET_COLS = list(raw_sample.columns[1:])
print(f"[STEP 8] id_col={ID_COL}  target_cols={TARGET_COLS}")


def find_test_image(image_id):
    cand = os.path.join(TEST_DIR, image_id)
    if os.path.exists(cand): return cand
    for ext in (".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp"):
        cand = os.path.join(TEST_DIR, image_id + ext)
        if os.path.exists(cand): return cand
    return None


def build_submission(apo_models, fasc_models, out_path, cache_tag):
    cache_path = os.path.join(CACHE_DIR, f"{cache_tag}.pkl")
    preds = {}
    if os.path.exists(cache_path):
        with open(cache_path, "rb") as f:
            preds = pickle.load(f)
        print(f"[STEP 8] resumed {len(preds)} cached predictions from {cache_path}")

    ids = raw_sample[ID_COL].astype(str).tolist()
    n_new = 0
    for i, image_id in enumerate(ids):
        if image_id in preds:
            continue
        img_path = find_test_image(image_id)
        if img_path is None:
            print(f"[STEP 8] WARNING: no test image found for id={image_id}, using fallback defaults")
            preds[image_id] = (15.0, 20.0 / math.sin(math.radians(15.0)), 20.0)
        else:
            apo_mask = predict_mask_ensemble(apo_models, img_path)
            fasc_mask = predict_mask_ensemble(fasc_models, img_path)
            pa, fl, mt = measure_geometry(apo_mask, fasc_mask, SCALE_MM_PER_PIXEL)
            preds[image_id] = (pa, fl, mt)
        n_new += 1

        if n_new % 50 == 0:
            with open(cache_path, "wb") as f:
                pickle.dump(preds, f)
            print(f"[STEP 8]   ...{i+1}/{len(ids)} images done, cache flushed, elapsed={elapsed_min():.1f}min")

    with open(cache_path, "wb") as f:
        pickle.dump(preds, f)

    rows = []
    for image_id in ids:
        pa, fl, mt = preds[image_id]
        row = {ID_COL: image_id}
        vals = [pa, fl, mt]
        for col, val in zip(TARGET_COLS, vals):
            row[col] = val
        rows.append(row)

    sub_df = pd.DataFrame(rows, columns=[ID_COL] + TARGET_COLS)
    sub_df.to_csv(out_path, sep=",", index=False)
    print(f"[STEP 8] wrote {out_path}  ({len(sub_df)} rows, comma-separated)")
    return sub_df

print('[STEP 8] submission builder ready')


## STEP 9 — Stage A: train the primary pair (ResNet34-UNet, apo + fasc)

In [ ]:

fasc_unet = ResNet34UNet(pretrained=True)
fasc_unet, fasc_unet_dice = train_model("fasc_unet", fasc_unet, fasc_train_loader, fasc_val_loader, epochs=30, patience=6)

apo_unet = ResNet34UNet(pretrained=True)
apo_unet, apo_unet_dice = train_model("apo_unet", apo_unet, apo_train_loader, apo_val_loader, epochs=30, patience=6)

print(f"[STEP 9] fasc_unet best val_dice={fasc_unet_dice:.4f}   apo_unet best val_dice={apo_unet_dice:.4f}")

_ = build_submission([apo_unet], [fasc_unet], os.path.join(WORK_DIR, "submission.csv"), cache_tag="stageA")
print("[STEP 9] CHECKPOINT: baseline (single-architecture) submission.csv written — safe to stop here if out of time.")


## STEP 10 — Stage B: train the diversity pair (ResNet50-DeepLabV3, apo + fasc)

In [ ]:

if time.time() < TRAIN_TIME_BUDGET_END:
    fasc_deeplab = DeepLabWrapper(build_deeplab(pretrained=True))
    fasc_deeplab, fasc_deeplab_dice = train_model("fasc_deeplab", fasc_deeplab, fasc_train_loader, fasc_val_loader, epochs=25, lr=5e-4, patience=6)

    apo_deeplab = DeepLabWrapper(build_deeplab(pretrained=True))
    apo_deeplab, apo_deeplab_dice = train_model("apo_deeplab", apo_deeplab, apo_train_loader, apo_val_loader, epochs=25, lr=5e-4, patience=6)

    print(f"[STEP 10] fasc_deeplab best val_dice={fasc_deeplab_dice:.4f}   apo_deeplab best val_dice={apo_deeplab_dice:.4f}")

    _ = build_submission([apo_unet, apo_deeplab], [fasc_unet, fasc_deeplab],
                          os.path.join(WORK_DIR, "submission.csv"), cache_tag="stageB")
    print("[STEP 10] CHECKPOINT: ensembled (UNet + DeepLabV3, TTA) submission.csv written. This is the final deliverable.")
else:
    print("[STEP 10] Time budget already exhausted after Stage A — keeping the Stage A baseline submission.csv as final.")


## STEP 11 — Final sanity check

In [ ]:

final_path = os.path.join(WORK_DIR, "submission.csv")
check_df = pd.read_csv(final_path, sep=",")
print(f"[STEP 11] final submission shape: {check_df.shape}")
print(f"[STEP 11] expected rows: {len(raw_sample)}  |  match: {len(check_df) == len(raw_sample)}")
print(f"[STEP 11] any NaNs: {check_df.isna().any().any()}")
print(check_df.head())
print(f"[STEP 11] DONE. Total wall-clock time: {elapsed_min():.1f} minutes.")
